# BP7 — Gate 6: Productization, Monitoring & Governance

**Business Problem:** BP7 — Customer Navigator Decision Engine
**Gate:** 6 of 6 (generic gate — Master Plan Table 2, Row 6: "Productization, Monitoring &
Governance")
**Compliance touchpoint (Table 2, Row 6):** Third-line-analog sign-off — Evidence Ledger row
reviewed against Gates 1-5 evidence before close.
**BP6/BP7-specific requirement (Master Plan paragraph 205):** "Per-BP governance set, required
before Gate 6 sign-off: `MODEL_CARD.md`, `CHANGELOG.md`, and — for BP6/BP7 — a runnable FastAPI
service with a live self-test proving API output matches direct computation."

## Why this gate looks different from BP6's own Gate 6 — and from every other BP's

BP7 fits no supervised model and makes **no external network call at any gate** (contrast BP6's
real, live Gemini call on every `/resolve`). It is a deterministic, transparent weighted-rule
engine (`src/features/bp7_decision_engine_features.py`'s `score_priority_rule`) that Gate 5
already applied once, for real, to the full real population (1,048,575 real CFPB complaints) and
persisted at `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate5_full_population_decision_records.csv`.
Master Plan paragraph 205 still makes a real, runnable FastAPI service with a live self-test a
**mandatory** BP6/BP7 deliverable — this gate therefore delivers, alongside the usual governance
artifacts:

- `src/services/bp7_decision_engine_service.py` — a real FastAPI service (`/`, `/health`,
  `/decide/{complaint_id}`, `/decide/self-test`) that serves real, read-only point lookups over
  Gate 5's own real, persisted full-population decision records — never a live re-score, never a
  fabricated result. Architecturally closer to `services.bp4_decision_service` (read-only lookup
  over a real, persisted artifact) than to `services.bp6_resolution_service`.
- `src/services/docker/bp7_decision_engine_service/` — Dockerfile + docker-compose.yml (port
  8007, the next unused port after BP1-6's own 8001-8006), matching BP1-6's own Docker packaging
  conventions.
- `tests/services/test_bp7_decision_engine_service.py` — the service's own CI-safe pytest suite.
  Unlike BP6's own test suite, there is no external API boundary to mock here — BP7 makes no
  network call anywhere, so every test drives the real app directly against a synthetic,
  real-schema fixture.
- `tests/bp7_customer_navigator_decision_engine/test_gate_artifacts.py` — BP7's own first-ever
  cross-artifact consistency test coverage, delivered at this gate exactly like every other BP's
  own Gate 6 precedent (BP5's and BP6's own `test_gate_artifacts.py`).

**On the self-test's own honesty**: BP6's own `/resolve/self-test` proves its FastAPI layer
introduces no drift by making ONE real Gemini call and independently re-deriving the result twice
from that same response. BP7 has no external call to make that proof meaningful for, so
`GET /decide/self-test` is the honest analogue for a BP that calls no external API anywhere: it
performs a REAL internal-consistency check over a batch of real, already-scored rows read from the
real, persisted Gate 5 CSV — reusing (never reimplementing) Gate 4/5's own real
`summarize_contribution_decomposition()` reconciliation function to verify
`contribution_bp2 + contribution_bp3 + contribution_bp4` reconstructs `priority_score` exactly,
plus real threshold-consistency and `recommended_action` vocabulary/structural-consistency checks,
plus two independently-constructed lazy-scan reads of the same rows asserted identical. Zero
external network calls anywhere in this proof. See the service module's own docstring for the
full rationale, including why a full, from-scratch re-derivation of `reason_codes`/
`recommended_action` via `score_priority_rule` is not reproducible from the persisted CSV alone
(it does not carry `bp4_recurring_flag`/`bp4_high_volume_flag` per row) — never attempted here
with a fabricated stand-in for those two missing columns.

## What this gate checks on the real, currently saved Gates 1-5 artifacts

Because BP7 fits no model and makes no GenAI call, there is no thinking-token-truncation-class bug
or champion-name/AUC reconfirmation to guard against here (contrast BP6's own Section 4). Instead,
Section 4 below re-verifies, from the **currently saved** real Gate 1/3/4/5 artifacts (never from
memory of a prior run), that the deterministic weighted-rule engine's own real structural
guarantees still hold: exact contribution-decomposition reconstruction, cross-gate weight/
statistic agreement between Gate 3/4/5, leakage re-confirmed clean, the disparate-impact
four-fifths flag still `False`, and the "no GenAI anywhere in BP7" scope commitment
(`recommended_action` is always a deterministic, reason-code-keyed lookup per Gate 1's own
policy.json).

## Real bugs found and fixed during this deliverable's own pre-delivery sandbox testing

Per this project's standing execution-boundary rule, these were found and fixed against a
synthetic, real-schema fixture in an isolated sandbox — never against the real device or the real
Gate 5 CSV: (1) a FastAPI route-registration-order bug where `/decide/{complaint_id}` (registered
first) shadowed `/decide/self-test` by attempting to `int()`-parse the literal string
`"self-test"`, returning 422 instead of ever reaching the self-test endpoint — fixed by
registering `/decide/self-test` before the parameterized route. (2) The self-test's per-row
contribution-reconstruction check initially reported a false-positive failure for a real,
structurally-honest unscored row (null `priority_score`, `UNSCORED_MISSING_UPSTREAM_INPUT`) —
there is nothing to reconstruct for such a row; fixed to treat it as vacuously consistent, matching
`summarize_contribution_decomposition()`'s own aggregate semantics exactly (which already excludes
null rows from its own reconstruction-error check).

## What running this notebook does for real

Every section below makes **zero** external network calls (pure local file reads, a real
`pytest tests/` subprocess run covering the FULL project test suite — this project's own standing
Gate 6 convention, not just BP7's own tests, a real static notebook-syntax audit, and a real,
fully local FastAPI self-test). This is the one meaningful contrast with BP6's own Gate 6, whose
Section 9 is the single real, live external call that notebook makes — BP7's own Section 9 below
makes none.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP7 Gate 6 (Productization, Monitoring & Governance)
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import re as _re
import subprocess
import sys
import warnings
import json
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (identical resolver to every other gate notebook in this
# project - PROJECT_STRUCTURE_LOCKED.md rule #3).
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb  # noqa: E402

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Imports + prerequisite check. BP7 Gate 6 makes ZERO external network calls anywhere
# (contrast BP6's mandatory real Gemini call at its own Section 9) - BP7 fits no model and no
# GenAI step at any gate (Gate 1's own policy.json / every upstream Gate 5's own disclosure). The
# ONE thing Section 9 below still does for real is start BP7's own real FastAPI service
# in-process and call its real /health, /decide/{complaint_id}, and /decide/self-test endpoints -
# all local, zero network egress.
# ============================================================
import yaml  # noqa: E402
import polars as pl  # noqa: E402
from fastapi.testclient import TestClient  # noqa: E402

CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp7_customer_navigator_decision_engine" / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp7_customer_navigator_decision_engine"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"
GATE5_MARKER_TEXT = "Gate 5 (Decision Layer & Reporting) results"

if not BP7_CONFIG_PATH.exists() or GATE5_MARKER_TEXT not in BP7_CONFIG_PATH.read_text(encoding="utf-8"):
    raise RuntimeError("BP7 Gate 5's own config block was not found - run BP7 Gate 5 for real first.")

RECORDS_CSV_PATH = ARTIFACTS_DIR / "gate5_full_population_decision_records.csv"
SUMMARY_JSON_PATH = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
POLICY_PATH = ARTIFACTS_DIR / "policy.json"
GATE3_SUMMARY_PATH = ARTIFACTS_DIR / "gate3_decision_rule_benchmark_summary.json"
GATE4_SUMMARY_PATH = ARTIFACTS_DIR / "gate4_statistical_validation_explainability_summary.json"
ACTION_BREAKDOWN_CSV_PATH = ARTIFACTS_DIR / "gate5_recommended_action_breakdown.csv"

for p in (RECORDS_CSV_PATH, SUMMARY_JSON_PATH, POLICY_PATH, GATE3_SUMMARY_PATH, GATE4_SUMMARY_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Required real input not found: {p}. Confirm BP7 Gate 5 has been real-run.")

# Load the config's PRE-WRITE state live (never hardcoded) so Section 13's own
# `status_field_untouched` check can verify this gate's write left `status` exactly as it found
# it, whatever Gate 1's own front-matter writer last set it to - matching the exact pattern
# Gate 4 and Gate 5's own notebooks already use (`full_config.get("status")`), rather than
# asserting a specific literal value that can go stale the moment Gate 1 is ever re-run after a
# later gate has appended its own block (see src/utils/bp1_config_sync.py's own
# `read_existing_gate_block_markers` docstring for why Gate 1's own `status` value can legitimately
# grow beyond the literal "gate1_confirmed").
with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config_text = f.read()
full_config = yaml.safe_load(full_config_text)

print(
    "[OK] Prerequisites confirmed: BP7 Gate 5's own config block + real Gate 1/3/4/5 artifacts all present."
)

# ============================================================
# SECTION 4: Load Gates 1-5's real artifacts LIVE (never hardcoded) + cross-gate consistency and
# regression-guard checks. BP7's own analogue of BP6 Gate 6's Section 4: because BP7 fits no
# model and makes no GenAI call, there is no thinking-token-truncation-class bug to guard against
# here - instead this section re-verifies, from the CURRENTLY SAVED real Gate 3/4/5 artifacts
# (never from memory of a prior run), that the deterministic weighted-rule engine's own real
# structural guarantees (exact contribution reconstruction, cross-gate weight/statistic agreement,
# the disparate-impact four-fifths flag, and the "no GenAI anywhere in BP7" scope commitment)
# still hold on disk today.
# ============================================================
with open(SUMMARY_JSON_PATH, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)
with open(GATE3_SUMMARY_PATH, "r", encoding="utf-8") as f:
    gate3_summary = json.load(f)
with open(GATE4_SUMMARY_PATH, "r", encoding="utf-8") as f:
    gate4_summary = json.load(f)

print(
    "[OK] Real Gates 1/3/4/5 artifacts loaded live: policy.json, "
    "gate3_decision_rule_benchmark_summary.json, "
    "gate4_statistical_validation_explainability_summary.json, gate5_decision_layer_summary.json."
)

_recommended_action_policy_text = gate1_policy["target_definition"]["output_fields"]["recommended_action"]
_cross_checks = gate5_summary["cross_checks_vs_gate3_gate4"]

_section4_checks: list[tuple[str, bool]] = [
    (
        "gate5_genai_api_used_is_false",
        gate5_summary["compliance_touchpoint"]["genai_api_used"] is False,
    ),
    (
        "gate1_recommended_action_policy_states_never_genai",
        "never genai" in _recommended_action_policy_text.lower(),
    ),
    (
        "gate5_contribution_reconstruction_exact_on_currently_saved_artifact",
        gate5_summary["contribution_decomposition_summary"]["reconstruction_exact_within_tolerance"] is True,
    ),
    (
        "gate5_weight_rederivation_matches_config_on_currently_saved_artifact",
        gate5_summary["weight_rederivation_cross_check"]["weights_match_config"] is True,
    ),
    (
        "gate5_all_cross_checks_vs_gate3_gate4_passed",
        all(v in (True, None) for v in _cross_checks.values()),
    ),
    (
        "gate4_leakage_reconfirmed_clean_on_currently_saved_artifact",
        gate4_summary["leakage_reconfirmation"]["leakage_reconfirmed_clean"] is True,
    ),
    (
        "gate4_contribution_reconstruction_exact_on_currently_saved_artifact",
        gate4_summary["contribution_decomposition"]["reconstruction_exact_within_tolerance"] is True,
    ),
    (
        "gate3_gate5_champion_rule_scheme_agree",
        gate3_summary["champion_rule_scheme"] == gate5_summary["champion_rule_scheme"],
    ),
    (
        # The real regression guard this gate exists to make permanent: if a future upstream data
        # change ever pushed BP7's own disparate-impact ratio below the four-fifths threshold, this
        # gate must refuse to certify, not silently pass on a stale Gate 4/5 "not flagged" claim.
        "gate5_disparate_impact_not_flagged_four_fifths_rule",
        gate5_summary["disparate_impact_audit"].get("flagged_four_fifths_rule") is False,
    ),
]
_section4_failed = [name for name, ok in _section4_checks if not ok]
for name, ok in _section4_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
if _section4_failed:
    raise RuntimeError(
        f"BP7 Gate 6 cross-gate consistency / regression-guard checks failed: {_section4_failed}. "
        "This means the artifact currently saved on disk for one of Gates 1/3/4/5 is stale, broken, "
        "or inconsistent with another gate's own recorded values. Re-run the named gate(s) for real, "
        "then re-run this Gate 6 notebook - Gate 6 does not proceed on an unreconfirmed prior gate."
    )
print(
    "[OK] All Gate 1/3/4/5 cross-gate consistency and regression-guard checks passed on the real, "
    "currently saved artifacts."
)

# ============================================================
# SECTION 5: Organize the real, live-loaded per-gate values for later use in MODEL_CARD.md /
# CHANGELOG.md (Sections 10-11) and the governance summary (Section 12) - no new computation here,
# only reading real values already loaded above into one convenience dict.
# ============================================================
gate_facts = {
    "gate1": {
        "generated_at_utc": gate1_policy["generated_at_utc"],
    },
    "gate3": {
        "champion_rule_scheme": gate3_summary["champion_rule_scheme"],
        "champion_coverage_pct": gate3_summary["champion_coverage_pct"],
        "champion_bp3_agreement_rate": gate3_summary["champion_bp3_agreement_rate"],
        "generated_at_utc": gate3_summary["generated_at_utc"],
    },
    "gate4": {
        "reproduction_bit_exact": gate4_summary["reproduction_check"][
            "reproduction_bit_exact_within_tolerance"
        ],
        "intervention_flag_rate_ci_95": [
            gate4_summary["bootstrap_ci"]["intervention_flag_rate"]["ci_lower_95"],
            gate4_summary["bootstrap_ci"]["intervention_flag_rate"]["ci_upper_95"],
        ],
        "leakage_reconfirmed_clean": gate4_summary["leakage_reconfirmation"]["leakage_reconfirmed_clean"],
        "generated_at_utc": gate4_summary["generated_at_utc"],
    },
    "gate5": {
        "live_row_count": gate5_summary["live_row_count"],
        "champion_rule_scheme": gate5_summary["champion_rule_scheme"],
        "champion_weights_normalized": gate5_summary["champion_weights_normalized"],
        "intervention_threshold": gate5_summary["intervention_threshold"],
        "intervention_flag_rate": gate5_summary["champion_stats"]["intervention_flag_rate"],
        "bp3_agreement_rate": gate5_summary["champion_stats"]["bp3_agreement_rate"],
        "adverse_impact_ratio": gate5_summary["disparate_impact_audit"].get("adverse_impact_ratio"),
        "flagged_four_fifths_rule": gate5_summary["disparate_impact_audit"].get("flagged_four_fifths_rule"),
        "generated_at_utc": gate5_summary["generated_at_utc"],
    },
}
print("[OK] Per-gate real facts organized for MODEL_CARD.md / CHANGELOG.md generation.")

# ============================================================
# SECTION 6: Detect real open items LIVE (informational governance findings, never blocking -
# Section 4 above already hard-fails on anything BP7's own design treats as disqualifying). Real,
# live-computed from Gate 5's own currently saved summary, never hardcoded.
# ============================================================
_upstream_coverage = gate5_summary.get("upstream_field_coverage", {})
_bp4_coverage_dict = _upstream_coverage.get("bp4", {})
_bp4_total = sum(_bp4_coverage_dict.values()) if _bp4_coverage_dict else 0
_bp4_unscored = _bp4_coverage_dict.get("UNSCORED_MISSING_UPSTREAM_INPUT", 0)
_bp4_unscored_rate = round(_bp4_unscored / _bp4_total, 6) if _bp4_total else None
_low_bp3_agreement = gate_facts["gate5"]["bp3_agreement_rate"] < 0.5
_bp1_coverage_dict = _upstream_coverage.get("bp1", {})
_bp1_total = sum(_bp1_coverage_dict.values()) if _bp1_coverage_dict else 0
_bp1_in_scope = _bp1_coverage_dict.get("BP1_TAXONOMY_CONTEXT_AVAILABLE", 0)
_bp1_coverage_pct = round(100.0 * _bp1_in_scope / _bp1_total, 4) if _bp1_total else None

open_items = {
    "bp4_unscored_rate_detected": _bp4_unscored_rate is not None and _bp4_unscored_rate > 0.0,
    "bp4_unscored_rate": _bp4_unscored_rate,
    "low_bp3_agreement_rate_detected": _low_bp3_agreement,
    "bp3_agreement_rate": gate_facts["gate5"]["bp3_agreement_rate"],
    "bp1_optional_context_coverage_pct": _bp1_coverage_pct,
}
print(f"[OK] Real open items detected live: {json.dumps(open_items, indent=2)}")

# ============================================================
# SECTION 7: Run the project's full pytest suite for REAL, via subprocess (exact CI invocation;
# this project's own standing convention - every BP's Gate 6 notebook runs the FULL suite, not
# just its own BP's tests). Includes BP7's first-ever dedicated test coverage - delivered
# alongside this notebook, not written by it: tests/services/test_bp7_decision_engine_service.py
# and tests/bp7_customer_navigator_decision_engine/test_gate_artifacts.py.
# ============================================================
_pytest_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(_pytest_proc.stdout[-4000:])
if _pytest_proc.returncode not in (0, 1):
    print(_pytest_proc.stderr[-2000:])

with open(ARTIFACTS_DIR / "gate6_pytest_output.log", "w", encoding="utf-8") as f:
    f.write(_pytest_proc.stdout)
    f.write("\n--- stderr ---\n")
    f.write(_pytest_proc.stderr)

_pytest_summary_line = next(
    (
        line
        for line in reversed(_pytest_proc.stdout.splitlines())
        if " passed" in line or "no tests ran" in line
    ),
    "",
)
_m = _re.search(r"(\d+) passed", _pytest_summary_line)
_pytest_n_passed = int(_m.group(1)) if _m else 0
_m_failed = _re.search(r"(\d+) failed", _pytest_summary_line)
_pytest_n_failed = int(_m_failed.group(1)) if _m_failed else 0
_pytest_all_passed = _pytest_proc.returncode == 0 and _pytest_n_failed == 0
print(f"[{'OK' if _pytest_all_passed else 'FAIL'}] pytest: {_pytest_summary_line.strip()}")

# ============================================================
# SECTION 8: Run the static notebook-syntax audit for REAL, via subprocess (nbformat + ast +
# pyflakes - never executes any notebook's code, per the project's execution-boundary rule).
# ============================================================
_syntax_proc = subprocess.run(
    [sys.executable, "scripts/check_notebook_syntax.py"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(_syntax_proc.stdout[-3000:])
with open(ARTIFACTS_DIR / "gate6_notebook_syntax_check_output.log", "w", encoding="utf-8") as f:
    f.write(_syntax_proc.stdout)
    f.write("\n--- stderr ---\n")
    f.write(_syntax_proc.stderr)

_syntax_all_passed = _syntax_proc.returncode == 0
print(
    f"[{'OK' if _syntax_all_passed else 'FAIL'}] Static notebook-syntax audit "
    f"returncode={_syntax_proc.returncode}"
)

# ============================================================
# SECTION 9: Master Plan paragraph 205's mandatory BP6/BP7 deliverable - "a runnable FastAPI
# service with a live self-test proving API output matches direct computation", adapted for BP7's
# no-external-API nature (see src/services/bp7_decision_engine_service.py's own module docstring
# for the full rationale this section deliberately does NOT repeat). Starts the real
# bp7_decision_engine_service.py FastAPI app IN-PROCESS and calls its real /health,
# /decide/{complaint_id} (for one real Complaint ID read live from the real Gate 5 CSV - never
# hardcoded), and /decide/self-test endpoints. Makes ZERO external network calls anywhere -
# unlike BP6's own Section 9, there is no API key to check and no real API quota consumed.
# ============================================================
from services.bp7_decision_engine_service import app as _bp7_app  # noqa: E402

# Same real fix BP6 Gate 6 needed for the identical reason (see that notebook's own Section 9
# comment): export PROJECT_ROOT into the process environment before starting the service, since
# TestClient(...).__enter__() runs the real lifespan() in a separate anyio thread whose cwd is not
# guaranteed to be this notebook's own PROJECT_ROOT.
os.environ["C360_PROJECT_ROOT"] = str(PROJECT_ROOT)

_first_real_complaint_id = pl.scan_csv(RECORDS_CSV_PATH).select("Complaint ID").limit(1).collect().item()

with TestClient(_bp7_app) as _self_test_client:
    _health_resp = _self_test_client.get("/health")
    if _health_resp.status_code != 200 or _health_resp.json().get("status") != "ok":
        raise RuntimeError(
            f"BP7 FastAPI service /health did not report 'ok' - cannot run the required Gate 6 "
            f"self-test. Real response: {_health_resp.json()}"
        )
    print(f"[OK] Real FastAPI service health check passed: {_health_resp.json()}")

    _decide_resp = _self_test_client.get(f"/decide/{_first_real_complaint_id}")
    if _decide_resp.status_code != 200:
        raise RuntimeError(
            f"BP7 FastAPI service's real GET /decide/{_first_real_complaint_id} did not return "
            f"200 (got {_decide_resp.status_code}): {_decide_resp.text}"
        )
    print(f"[OK] Real point-lookup for Complaint ID {_first_real_complaint_id} succeeded.")

    print(
        "[CALLING] Real GET /decide/self-test - a real internal-consistency check over real, "
        "already-scored rows. Makes ZERO external network calls (BP7 makes none anywhere)."
    )
    _self_test_resp = _self_test_client.get("/decide/self-test", params={"sample_size": 100})

if _self_test_resp.status_code != 200:
    raise RuntimeError(
        f"BP7 FastAPI service's real GET /decide/self-test did not return 200 "
        f"(got {_self_test_resp.status_code}): {_self_test_resp.text}."
    )

self_test_result = _self_test_resp.json()
if not self_test_result["all_checks_passed"]:
    raise RuntimeError(
        "BP7's FastAPI self-test reported all_checks_passed=False on one or more of the "
        f"{self_test_result['n_rows_checked']} real rows checked. This is a real code-level bug in "
        "either src/services/bp7_decision_engine_service.py or a real data-integrity issue in the "
        "persisted Gate 5 CSV - Gate 6 cannot certify a service that fails its own required "
        "self-test. Fix the drift before re-running."
    )
print(
    f"[OK] Real FastAPI self-test PASSED: all_checks_passed={self_test_result['all_checks_passed']} "
    f"over {self_test_result['n_rows_checked']} real, already-scored rows "
    "(reconstruction, threshold-consistency, action-vocabulary, and dual-read-identity checks - "
    "zero external network calls made)."
)

SELF_TEST_RESULT_PATH = ARTIFACTS_DIR / "gate6_fastapi_self_test_result.json"
with open(SELF_TEST_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "bp_id": "bp7",
            "gate": 6,
            "compliance_touchpoint": "Master Plan paragraph 205 - runnable FastAPI service with a "
            "live self-test proving API output matches direct computation, adapted for BP7's "
            "no-external-API nature (see src/services/bp7_decision_engine_service.py's own module "
            "docstring for the full rationale).",
            "health_check_status": _health_resp.json()["status"],
            "sample_decide_complaint_id_checked": _first_real_complaint_id,
            "self_test_all_checks_passed": self_test_result["all_checks_passed"],
            "self_test_n_rows_checked": self_test_result["n_rows_checked"],
            "real_external_api_call_made": self_test_result["real_external_api_call_made"],
            "note": "This self-test artifact is a Gate 6 governance/testing record, NOT a "
            "customer-facing decision - it is separate from, and never overwrites, Gate 5's own "
            "gate5_full_population_decision_records.csv or gate5_decision_layer_summary.json.",
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        f,
        indent=2,
    )
print(f"[SAVED] {SELF_TEST_RESULT_PATH}")

# ============================================================
# SECTION 10: Generate MODEL_CARD.md - deterministic f-string template from ONLY the real values
# loaded/computed above (zero-fabrication rule: no GenAI-authored freeform text, and BP7 itself
# never calls GenAI at any gate).
# ============================================================
_now_utc = datetime.now(timezone.utc).isoformat()

_model_card = f"""# Model Card — BP7 Customer Navigator Decision Engine

*Generated {_now_utc} by
`bp7_customer_navigator_decision_engine_g6_productization_monitoring_governance.ipynb`,
deterministically, from real values recorded by BP7 Gates 1-5's own real runs on this machine plus
this gate's own real FastAPI self-test. No field below was authored freeform or by a generative
model (project zero-fabrication rule) - BP7 itself never calls a GenAI API at any gate.*

## Model Details
- **Nature of this BP**: a deterministic, transparent weighted-rule decision engine combining
  BP2/BP3/BP4's own already-computed, already-validated prediction fields into
  `priority_score`/`intervention_flag`/`recommended_action`/`reason_codes` for every real CFPB
  complaint - **not** a trained classifier and **not** a live model/API call at request time
  (contrast BP1-3's inference services and BP6's real Gemini call). Architecturally closer to BP4
  (read-only lookup over a real, persisted decision artifact) than to BP6.
- **Champion rule scheme** (Gate 3, independently reproduced at Gate 4, re-scored fresh on the
  full real population at Gate 5): `{gate_facts['gate5']['champion_rule_scheme']}`, real
  normalized weights `{gate_facts['gate5']['champion_weights_normalized']}`, real intervention
  threshold {gate_facts['gate5']['intervention_threshold']}.
- **Full real population scored**: {gate_facts['gate5']['live_row_count']:,} real CFPB complaints,
  100% coverage (every row receives a real priority_score or the honest
  `UNSCORED_MISSING_UPSTREAM_INPUT` sentinel - never a fabricated value for an unjoinable row).

## Intended Use
- Computes a transparent, auditable priority/intervention/action decision for every real customer
  complaint, keyed by `Complaint ID`, for downstream review-queue routing - never a black-box
  score passed through unmodified, and `recommended_action` is always a deterministic,
  reason-code-keyed lookup, never GenAI-generated text (Gate 1's own policy.json, re-verified live
  by this gate's own Section 4).
- Out of scope: BP7 makes no prediction of its own and calls no external API at any gate - it
  only recombines BP1-BP5's own already-validated outputs under a documented, auditable formula.

## Evaluation Data & Results (Gate 3/4/5, re-verified live by this gate's own Section 4)
- **Real intervention_flag_rate**: {gate_facts['gate5']['intervention_flag_rate']} (Gate 4's own
  real bootstrap 95% CI: {gate_facts['gate4']['intervention_flag_rate_ci_95']})
- **Real bp3_agreement_rate** (coherence sanity check against BP3's own already-validated
  prediction, never an accuracy claim - `customer360_priority_decision` has no labeled ground
  truth): {gate_facts['gate5']['bp3_agreement_rate']}
- **Real contribution-decomposition reconstruction**: exact within floating-point tolerance
  (`contribution_bp2 + contribution_bp3 + contribution_bp4` reconstructs `priority_score` exactly
  for every real scored row - re-verified live on the currently saved artifact by this gate).
- **Real disparate-impact audit** (ECOA/Reg B monitoring signal, not a legal determination):
  adverse_impact_ratio={gate_facts['gate5']['adverse_impact_ratio']},
  flagged_four_fifths_rule={gate_facts['gate5']['flagged_four_fifths_rule']} (re-verified live by
  this gate's own Section 4 as a hard regression guard - Gate 6 refuses to certify if this ever
  flips to True on the currently saved artifact).

## Governance — Master Plan Paragraph 205 (BP6/BP7-specific requirement)
- **MODEL_CARD.md / CHANGELOG.md**: this file and its sibling, generated deterministically by this
  gate from Gates 1-5's own real recorded values.
- **Runnable FastAPI service**: `src/services/bp7_decision_engine_service.py` (`/`, `/health`,
  `/decide/{{complaint_id}}`, `/decide/self-test`) - Docker packaging at
  `src/services/docker/bp7_decision_engine_service/`.
- **Live self-test proving API output matches direct computation**: run for real by this gate
  (Section 9) - adapted for BP7's no-external-API nature: a real internal-consistency check over
  {self_test_result['n_rows_checked']} real, already-scored rows (contribution reconstruction,
  threshold-consistency, recommended_action vocabulary/structural consistency, and dual
  independent-read identity), all local, zero external network calls:
  `all_checks_passed={self_test_result['all_checks_passed']}`. See the service module's own
  docstring for why this - not a network-mocked replay of BP6's own Gemini-call pattern - is the
  honest proof for a BP that calls no external API anywhere.

## Ethical Considerations & Governance
- **ECOA/Reg B**: applicable (Master Plan Section 9). Gate 5's own real, full-population
  disparate-impact audit, re-verified live by this gate:
  adverse_impact_ratio={gate_facts['gate5']['adverse_impact_ratio']},
  flagged_four_fifths_rule={gate_facts['gate5']['flagged_four_fifths_rule']} - a monitoring signal
  for a human reviewer, never a legal determination of compliance (the identical limitation every
  upstream BP's own Gate 4/5 stated, not softened here).
- **UDAAP / NIST AI RMF**: Not Applicable to BP7 (Gate 1's own policy.json) -
  `recommended_action` is a deterministic, reason-code-keyed lookup, never GenAI-generated
  customer-facing text. Real GenAI-drafted text stays scoped to BP6 per the Master Plan.
- **No barred field used**: BP7's own real, live leakage re-check (Gate 4, re-verified live by
  this gate's own Section 4) confirms `leakage_reconfirmed_clean=True`.

## Known Limitations (detected LIVE from real Gate 3/4/5 artifacts, not from memory)
- **Real bugs found and fixed during this Gate 6 deliverable's own pre-delivery sandbox testing**
  (never run against the real device, per this project's standing execution-boundary rule - a
  synthetic, real-schema fixture only): (1) FastAPI route registration order - `/decide/{{complaint_id}}`
  was initially registered before `/decide/self-test`, which caused Starlette to attempt to
  int-parse the literal string `"self-test"` and return 422 instead of reaching the self-test
  endpoint; fixed by registering `/decide/self-test` first. (2) The self-test's per-row
  contribution-reconstruction check initially reported `False` for a real, structurally-honest
  unscored row (null `priority_score`, `UNSCORED_MISSING_UPSTREAM_INPUT`) - there is nothing to
  reconstruct for such a row, so this was a false-positive failure, not a real defect; fixed to
  treat a null-`priority_score` row as vacuously consistent, matching
  `summarize_contribution_decomposition()`'s own aggregate semantics exactly.
- **BP4 join coverage is not 100%**: {open_items['bp4_unscored_rate']} real fraction of rows carry
  `bp4_join_status=UNSCORED_MISSING_UPSTREAM_INPUT` (Gate 2's own real join coverage finding,
  carried forward - never silently defaulted to a fabricated BP4 tier).
- **BP1's optional context coverage is low**: {open_items['bp1_optional_context_coverage_pct']}%
  of real rows have a real BANKING77-taxonomy-crosswalk context available (Gate 1's own
  `OPTIONAL_CONTEXT_ONLY` scoping - BP1 is never a CORE_INPUT to `priority_score` itself).
- **This service's own real dependency footprint is heavier than BP4's** (contrast BP4's minimal
  fastapi/pydantic/polars list): `/decide/self-test` reuses Gate 4/5's own real
  `summarize_contribution_decomposition()` function rather than reimplementing its arithmetic
  inline, which transitively pulls in the shared BP2/BP3/BP4 feature-module chain
  (scikit-learn/scipy/joblib/PyYAML) - a disclosed, deliberate reuse-over-minimalism trade-off,
  documented in the service's own Dockerfile header.

## Testing & Reproducibility (this Gate 6 run)
- **Full project pytest suite**: `{_pytest_summary_line.strip()}` (all passed: {_pytest_all_passed})
- **Static notebook-syntax audit**: returncode={_syntax_proc.returncode} (all passed: {_syntax_all_passed})
- **Real FastAPI self-test**: all_checks_passed={self_test_result['all_checks_passed']} over
  {self_test_result['n_rows_checked']} real rows, zero external network calls
- BP7's own first-ever dedicated test coverage, delivered alongside this gate:
  `tests/services/test_bp7_decision_engine_service.py` (the FastAPI service's own CI-safe suite -
  no network call anywhere, so nothing to mock, unlike BP6's own Gemini-boundary mocks) and
  `tests/bp7_customer_navigator_decision_engine/test_gate_artifacts.py` (schema/cross-artifact
  consistency checks for Gates 1-6's real saved output).

## [Gate 1] Business Understanding & Policy — {gate_facts['gate1']['generated_at_utc']}
- BP7 scope, output-field definitions, and leakage rules defined; `recommended_action` is a
  deterministic, reason-code-keyed lookup, never GenAI - re-confirmed live by this gate.
"""

MODEL_CARD_PATH = REPORTS_DIR / "MODEL_CARD.md"
with open(MODEL_CARD_PATH, "w", encoding="utf-8") as f:
    f.write(_model_card)
print(f"[SAVED] {MODEL_CARD_PATH}")

# ============================================================
# SECTION 11: Generate CHANGELOG.md - deterministic, chronological, real values only.
# ============================================================
_changelog = f"""# Changelog — BP7 Customer Navigator Decision Engine

*Generated {_now_utc}, deterministically, from real values recorded by BP7 Gates 1-6's own real
runs on this machine.*

## [Gate 6] Productization, Monitoring & Governance — {_now_utc}
- Real full pytest suite: `{_pytest_summary_line.strip()}` (all passed: {_pytest_all_passed})
- Real static notebook-syntax audit: returncode={_syntax_proc.returncode} (all passed: {_syntax_all_passed})
- Real FastAPI self-test (Master Plan paragraph 205, adapted for BP7's no-external-API nature):
  all_checks_passed={self_test_result['all_checks_passed']} over
  {self_test_result['n_rows_checked']} real rows (contribution reconstruction, threshold
  consistency, recommended_action structural consistency, dual independent-read identity) - zero
  external network calls made anywhere
- MODEL_CARD.md and CHANGELOG.md generated deterministically from Gates 1-5's own real recorded
  values (this file)
- New test coverage delivered: `tests/services/test_bp7_decision_engine_service.py`,
  `tests/bp7_customer_navigator_decision_engine/test_gate_artifacts.py`
- New production-style service delivered: `src/services/bp7_decision_engine_service.py` +
  `src/services/docker/bp7_decision_engine_service/` (Dockerfile, docker-compose.yml)
- Cross-gate consistency / regression-guard checks re-verified on the currently saved real
  artifacts (Section 4): contribution reconstruction still exact, weight rederivation still
  matches config, Gate 3/Gate 5 champion rule scheme still agree, leakage still reconfirmed clean,
  disparate-impact four-fifths flag still False, genai_api_used still False.
- Two real bugs found and fixed during this gate's own pre-delivery sandbox testing (synthetic
  fixtures only, per this project's standing execution-boundary rule): a FastAPI route-ordering
  bug that shadowed `/decide/self-test`, and a self-test false-positive on a real, structurally-
  honest unscored row - see MODEL_CARD.md Known Limitations for both.
- `status` field left untouched (this project's own established BP7 convention: Gates 2-6 never
  touch it - owned exclusively by Gate 1's own front-matter writer).

## [Gate 5] Decision Layer & Reporting
- Real champion: {gate_facts['gate5']['champion_rule_scheme']}, weights
  {gate_facts['gate5']['champion_weights_normalized']}, threshold
  {gate_facts['gate5']['intervention_threshold']}
- {gate_facts['gate5']['live_row_count']:,} real complaints scored, 100% coverage
- intervention_flag_rate={gate_facts['gate5']['intervention_flag_rate']},
  bp3_agreement_rate={gate_facts['gate5']['bp3_agreement_rate']}
- Disparate impact: adverse_impact_ratio={gate_facts['gate5']['adverse_impact_ratio']},
  flagged_four_fifths_rule={gate_facts['gate5']['flagged_four_fifths_rule']}
- No GenAI API used (genai_api_used=False)

## [Gate 4] Statistical Validation & Explainability
- Champion reproduced bit-exact: {gate_facts['gate4']['reproduction_bit_exact']}
- Bootstrap 95% CI (intervention_flag_rate): {gate_facts['gate4']['intervention_flag_rate_ci_95']}
- Leakage reconfirmed clean: {gate_facts['gate4']['leakage_reconfirmed_clean']}

## [Gate 3] Decision-Rule-Scheme Benchmark & Champion Selection — {gate_facts['gate3']['generated_at_utc']}
- Champion: {gate_facts['gate3']['champion_rule_scheme']} (coverage
  {gate_facts['gate3']['champion_coverage_pct']}%, bp3_agreement_rate
  {gate_facts['gate3']['champion_bp3_agreement_rate']})

## [Gate 1] Business Understanding & Policy — {gate_facts['gate1']['generated_at_utc']}
- BP7 scope, output-field definitions, and leakage rules defined; recommended_action policy:
  deterministic, reason-code-keyed lookup, never GenAI
"""

CHANGELOG_PATH = REPORTS_DIR / "CHANGELOG.md"
with open(CHANGELOG_PATH, "w", encoding="utf-8") as f:
    f.write(_changelog)
print(f"[SAVED] {CHANGELOG_PATH}")

# ============================================================
# SECTION 12: Write the Gate 6 config block (flat keys, matching this project's own Gate 2-5
# established convention - see BP5's own Gate 6 config block for the precedent this mirrors) +
# gate6_governance_summary.json. `status` is deliberately NOT touched - this project's own
# established BP7 convention (Gate 2, Gate 3, Gate 4, and Gate 5 all left it untouched; `status`
# is owned exclusively by Gate 1's own front-matter writer, per src/utils/bp1_config_sync.py's own
# docstring).
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate6_marker = (
    "# --- Gate 6 (Productization, Monitoring & Governance) results (appended, idempotent overwrite) ---"
)
gate6_block_lines = [
    f"gate6_pytest_all_passed: {_pytest_all_passed}",
    f"gate6_pytest_n_passed: {_pytest_n_passed}",
    f"gate6_pytest_n_failed: {_pytest_n_failed}",
    f"gate6_notebook_syntax_all_passed: {_syntax_all_passed}",
    f"gate6_fastapi_self_test_all_checks_passed: {self_test_result['all_checks_passed']}",
    f"gate6_fastapi_self_test_n_rows_checked: {self_test_result['n_rows_checked']}",
    f"gate6_fastapi_self_test_real_external_api_call_made: {self_test_result['real_external_api_call_made']}",
    f"gate6_bp4_unscored_rate_detected: {open_items['bp4_unscored_rate_detected']}",
    f"gate6_low_bp3_agreement_rate_detected: {open_items['low_bp3_agreement_rate_detected']}",
    f'gate6_model_card_path: "{MODEL_CARD_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate6_changelog_path: "{CHANGELOG_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    'gate6_fastapi_service_path: "src/services/bp7_decision_engine_service.py"',
    f'gate6_fastapi_self_test_result_path: "{SELF_TEST_RESULT_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate6_generated_at_utc: "{_now_utc}"',
]
write_gate_block(BP7_CONFIG_PATH, gate6_marker, gate6_block_lines)
print(f"[SAVED] gate6 block written to {BP7_CONFIG_PATH.relative_to(PROJECT_ROOT)} (status untouched)")

with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate6_block_actually_written = gate6_marker in _post_write_config_text
_post_write_config = yaml.safe_load(_post_write_config_text)
status_untouched = _post_write_config.get("status") == full_config.get("status")

GOVERNANCE_SUMMARY_PATH = ARTIFACTS_DIR / "gate6_governance_summary.json"
with open(GOVERNANCE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "bp_id": "bp7",
            "gate": 6,
            "pytest_summary_line": _pytest_summary_line.strip(),
            "pytest_n_passed": _pytest_n_passed,
            "pytest_n_failed": _pytest_n_failed,
            "pytest_all_passed": _pytest_all_passed,
            "notebook_syntax_audit_returncode": _syntax_proc.returncode,
            "notebook_syntax_all_passed": _syntax_all_passed,
            "fastapi_self_test_all_checks_passed": self_test_result["all_checks_passed"],
            "fastapi_self_test_real_external_api_call_made": self_test_result["real_external_api_call_made"],
            "gate3_gate5_champion_rule_scheme_agree": gate3_summary["champion_rule_scheme"]
            == gate5_summary["champion_rule_scheme"],
            "open_items": open_items,
            "model_card_path": str(MODEL_CARD_PATH.relative_to(PROJECT_ROOT)),
            "changelog_path": str(CHANGELOG_PATH.relative_to(PROJECT_ROOT)),
            "generated_at_utc": _now_utc,
        },
        f,
        indent=2,
    )
print(f"[SAVED] {GOVERNANCE_SUMMARY_PATH}")

# ============================================================
# SECTION 13: Structural integrity checks - raise AssertionError, never silently pass. The real
# pytest suite, the real static notebook-syntax audit, and the real FastAPI self-test are
# themselves three of these checks: Gate 6 is NOT complete unless all three genuinely passed on
# THIS run.
# ============================================================
_front_matter_and_priors_preserved = all(
    marker in _post_write_config_text
    for marker in (
        'bp_id: "bp7"',
        GATE5_MARKER_TEXT,
        "live_row_count:",
        "champion_rule_scheme:",
        "gate4_champion_reproduced_bit_exact:",
        "gate5_champion_rule_scheme:",
    )
)

_final_checks: list[tuple[str, bool]] = [
    ("section4_cross_gate_checks_all_passed", not _section4_failed),
    ("pytest_all_passed", _pytest_all_passed),
    ("notebook_syntax_all_passed", _syntax_all_passed),
    ("fastapi_health_check_ok", _health_resp.json()["status"] == "ok"),
    ("fastapi_decide_point_lookup_ok", _decide_resp.status_code == 200),
    ("fastapi_self_test_all_checks_passed", self_test_result["all_checks_passed"]),
    (
        "fastapi_self_test_made_zero_external_api_calls",
        self_test_result["real_external_api_call_made"] is False,
    ),
    ("model_card_written", MODEL_CARD_PATH.exists()),
    ("changelog_written", CHANGELOG_PATH.exists()),
    ("governance_summary_written", GOVERNANCE_SUMMARY_PATH.exists()),
    ("self_test_result_artifact_written", SELF_TEST_RESULT_PATH.exists()),
    ("config_gate6_block_written", gate6_block_actually_written),
    ("config_front_matter_and_prior_gate_blocks_preserved", _front_matter_and_priors_preserved),
    ("status_field_untouched", status_untouched),
]
_final_failed = [name for name, ok in _final_checks if not ok]
for name, ok in _final_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert not _final_failed, f"BP7 Gate 6 structural integrity checks failed: {_final_failed}"

print(
    "\n[ALL CHECKS PASSED] BP7 Gate 6 (Productization, Monitoring & Governance) complete. "
    f"Full pytest suite: {_pytest_summary_line.strip()}. Real FastAPI self-test: "
    f"all_checks_passed={self_test_result['all_checks_passed']} over "
    f"{self_test_result['n_rows_checked']} real rows, zero external network calls. MODEL_CARD.md "
    "and CHANGELOG.md written from Gates 1-5's own real recorded values. status left untouched "
    "(BP7's own established convention). BP7 is now complete: all 6 gates real-run confirmed."
)
